In [2]:
import os
import ultralytics
from ultralytics import YOLO
import torch
import numpy as np
from skimage.measure import label, regionprops, regionprops_table
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics.yolo.utils.plotting import Annotator
torch.cuda.is_available()
ultralytics.checks()

Ultralytics YOLOv8.0.146  Python-3.9.17 torch-2.0.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 8192MiB)
Setup complete  (20 CPUs, 31.7 GB RAM, 379.7/943.7 GB disk)


### Create dataframe based from the predicted image

In [82]:
#helper function

def count_plant(dic,inference,name,t_p,condition):
    """
    Count the number of plants detected in a image and compare it with the True number of plants.
    
    input: dictonary
           inference: predicted mask and bounding box
           image name: image id
           t_p: Expected plants
           condition: single or split root
    
    output: dictionary contains the image id and correcponding valus.
    
    """
    dic['image'].append(name)
    dic['Condition'].append(condition)
    dic['T_plant'].append(t_p)
    dic['P_plant'].append(inference.shape[0])
    
    return dic


def AMM(mask,bbx):
    ""
    x1,y1,x2,y2=bbx
    width = x2 - x1
    height = y2 - y1
    
    major_axis=max(width,height)
    minor_axis=min(width,height)
    
    area=np.sum(mask)
    
    return major_axis,minor_axis,area,y1

def centroid(box):
    x1,y1,x2,y2=box
    cen_x=x1
    cen_y=y1
    return cen_x,cen_y
#round1 plant arrangement
c1=[[3,3],[6,2],[2,2],[5,1],[1,1],[4,4]]
c2=[[9,1],[12,3],[8,4],[11,2],[7,3],[10,4]]
##round2 plant arrangement
c12=[[10,4],[7,3],[11,2],[8,4],[12,3],[9,1]]
c22=[[4,4,],[1,1],[5,1],[2,2],[6,2],[3,3]]

def arrange(frame,c,img_name,condition):
    """
    Arrange the image parameter according to the Plant ID and treatment.
    input: data frame, c1 =[plot ID, treatment], image name
    
    output: dataframe with proper ID and treatment assigned to it.
    
    """
    
    new_frame=frame.copy()
    new_frame['Plot']= 0
    new_frame['Treatment']= 0
    new_frame=new_frame.sort_values(['y'],ascending=[True])
    new_frame.reset_index(inplace=True)
    i=0
    while i< new_frame.shape[0]//2:
        
        if new_frame.iloc[i+i,-4]>new_frame.iloc[i+i+1,-4]:
            #print('first one bigger')
            #print(c1[i+i+1][0],c1[i+i+1][1],c1[i+i][0],c1[i+i][1])
            new_frame.loc[i+i,'Plot']=c[i+i+1][0]
            new_frame.loc[i+i,'Treatment']= c[i+i+1][1]
            new_frame.loc[i+i+1,'Plot']=c[i+i][0]
            new_frame.loc[i+i+1,'Treatment']= c[i+i][1]
        else:
#             print('first one smaller')
#             print(c1[i+i][0],c1[i+i][1],c1[i+i+1][0],c1[i+i+1][1],f'i={i}')
            new_frame.loc[i+i,'Plot']=c[i+i][0]
            new_frame.loc[i+i,'Treatment']= c[i+i][1]
            new_frame.loc[i+i+1,'Plot']=c[i+i+1][0]
            new_frame.loc[i+i+1,'Treatment']= c[i+i+1][1]
            
    
        i+=1
    
    new_frame['Image']=img_name
    new_frame['Condition']=condition
    return new_frame.drop(['index','x','y'],axis=1)


#single root round1
c3=[[1,1],[2,2],[3,3],[4,4],[5,1],[6,2],[7,3],[8,4],[9,1],[10,4],[11,2],[12,3]]

#round2
cl32=[[3,2],[2,4],[1,1],[4,3],[5,1],[6,2],[9,1],[8,2],[7,3],[10,4],[11,3],[12,4]]
def arrange2(frame,ci,img_name,condition):
    """
    Arrange the image parameter according to the Plant ID and treatment.
    input: data frame, c1 =[plot ID, treatment], image name,condition
    
    output: dataframe with proper ID and treatment assigned to it.
    
    """
    
    new_frame=frame.copy()
    new_frame['Plot']= 0
    new_frame['Treatment']= 0
    new_frame=new_frame.sort_values(['y'],ascending=[True])
    new_frame.reset_index(inplace=True)
    new_frame.drop('index',axis=1,inplace=True)
    i=0
    step=0
    idx=0
    df=[]
    while i< new_frame.shape[0]//3:
        
        new_frame1=new_frame[i+step-i:3+step].sort_values(['x'])#sort the data frame segment (low to high)
        new_frame1.reset_index(inplace=True)
        new_frame1.drop('index',axis=1,inplace=True)
        #print(f'Vlaues would be{ci[i+step-i:3+step]}')
        for l,k in enumerate(ci[i+step-i:3+step]): # assign plot and treatment from low to high
            new_frame1.iloc[l,-2]=k[0]
            new_frame1.iloc[l,-1]=k[1]
    
        i+=1
        step+=3
        df.append(new_frame1)
    
    new_frame=pd.concat(df)
    print(img_name)
    new_frame['Image']=img_name
    new_frame['Condition']=condition
    new_frame.reset_index(inplace=True)
    return new_frame.drop(['index','x','y'],axis=1)


def pred(img,model,value):
    """
    input: image path and model path
    
    output: image parameter (major axis, minor axis, area) as dataframe
    
    """
    #print(name)
    orig_img= cv2.imread(img)
    x_scale,y_scale=round(orig_img.shape[1]/640,4),round(orig_img.shape[0]/640,4)
    orig_img=cv2.resize(orig_img,(640,640))
    
    results = model.predict(source=orig_img.copy(),conf=0.5)
    orig_img=cv2.resize(orig_img,None,fy=y_scale,fx=x_scale, interpolation=cv2.INTER_CUBIC)
    region=[]
    for result in results:
        masks=result.masks.cpu().data.numpy()
        #masks= mask.masks.astype(bool)
        boxes=results[0].boxes.cpu().numpy()
        
        for i,v in enumerate(zip(masks,boxes)):
            m,box=v
            r = box.xyxy[0].astype(int)
            r[0],r[1]=int(r[0]*x_scale),int(r[1]*y_scale)
            r[2],r[3]=int(r[2]*x_scale),int(r[3]*y_scale)
            m=cv2.resize(m,None,fy=round(y_scale,4),fx=round(x_scale,4), interpolation=cv2.INTER_CUBIC)
            m=m.astype(bool)
            img=Image.fromarray(np.uint8(orig_img))
            cropped_image = img.crop(r)
            
            cropped_mask = Image.fromarray(np.uint8(m)).crop(r)
            arry=np.array(cropped_mask)
            
            df=pd.DataFrame(regionprops_table(arry.astype(int)*255,properties=('axis_major_length',
                                                 'axis_minor_length','area')))
    
            df['x']=r[0]
            df['y']=r[1]
            region.append(df)
    d1=pd.concat(region)
    d1.reset_index(inplace=True)
    d1=d1.sort_values(by='area',ascending=False)
    if d1.shape[0]>value:
        d1=d1.iloc[0:value,:]
    
    return d1


In [83]:
#go pro data
def makeFile(imgPath,modelPath,value,save,c):
    """
    get images from the path then perform prediction over the images.
    This predicted result are converted into dataframe for each images and stack them at the end
    
    input
        imgpath: location of image
        modelpath: location of trained custom model
        value: maximum number of instance in each each. 
                6:  split root
                12: single root
        save: {1: to save dataframe,
                0: not save}
                
    output
        dataframe contains the area, major_axis, minor_axis, image name
    
    """
    
    images=os.listdir(imgPath)
    model=YOLO(modelPath)
    count={0:0, 1:0 }
    tracking={'image':[],'T_plant':[],'P_plant':[],'Condition':[]}
    f=[]
    for i in images:
        print(f'processing {os.path.join(imgPath,i)}')
        print(i.split('_')[-1])
        if i.split('_')[1]=='c1':
#             name='Camera 1'+i.split('_')[-1]
            name=i
            r=pred(os.path.join(imgPath,i),model,value)
            #track the number of plant detected in each image and compare it with the expected value
            tracking=count_plant(tracking,r,name,6,'split root')
            f.append(arrange(r,c[0],name,'split root'))
            count[0]+=1
        elif i.split('_')[1]=='c2':
#             name='Camera 2'+ i.split('_')[-1]
            name=i
            r=pred(os.path.join(imgPath,i),model,value)
             #track the number of plant detected in each image and compare it with the expected value
            tracking=count_plant(tracking,r,name,6,'split root')
            f.append(arrange(r,c[1],name,'split root'))
            
            count[0]+=1
        else:
#             name='Camera 3'+ i.split('_')[-1]
            name=i
            r=pred(os.path.join(imgPath,i),model,12)
            tracking=count_plant(tracking,r,name,12,'single root')
            f.append(arrange2(r,c[2],name,'single root'))
            count[1]+=1
        
       
    print(f'Total split {count[0]} \n Total single {count[1]}')
    if save==0:
        return pd.concat(f),tracking
    else:
        return pd.concat(f).to_csv('done.csv')

##for pi data
def makeFilepi(imgPath,modelPath,value,save):
    """
    get images from the path then perform prediction over the images.
    This predicted result are converted into dataframe for each images and stack them at the end
    
    input
        imgpath: location of image
        modelpath: location of trained custom model
        value: maximum number of instance in each each. 
                6:  split root
                12: single root
        save: {1: to save dataframe,
                0: not save}
                
    output
        dataframe contains the area, major_axis, minor_axis, image name
    
    """
    
    images=os.listdir(imgPath)
    model=YOLO(modelPath)
    count={0:0, 1:0 }
    tracking={'image':[],'T_plant':[],'P_plant':[],'Condition':[]}
    f=[]
    for i in images:
        print(f'processing {os.path.join(imgPath,i)}')
        #print(i.split('_')[-1])
        if i.split('_')[1]=='c1':
            name=i
            r=pred(os.path.join(imgPath,i),model,value)
            tracking=count_plant(tracking,r,name,6,'split root')
            f.append(arrange(r,c12,name,'split root'))
            count[0]+=1
        elif i.split('_')[1]=='c2':
            name=i
            r=pred(os.path.join(imgPath,i),model,value)
            tracking=count_plant(tracking,r,name,6,'split root')
            f.append(arrange(r,c22,name,'split root'))
            
            count[0]+=1
        else:
            name= i
            r=pred(os.path.join(imgPath,i),model,12)
            tracking=count_plant(tracking,r,name,12,'single root')
            f.append(arrange2(r,cl32,name,'single root'))
            count[1]+=1
        
       
    print(f'Total split {count[0]} \n Total single {count[1]}')
    if save==0:
        return pd.concat(f),tracking
    else:
        pd.concat(f).to_csv('s_r1_on_r2.csv')

In [131]:
#making file with function
c_r1=[c1,c2,c3]
c_r2=[c12,c22,cl32]
#image path
imgPathr1='work_df/raw data/round 1/images'
imgPathr2='work_df/raw data/round22(renamed)'
imgPath70='work_df/training data/dataset70/test/images'
pi06='work_df/pi2/common data/06'
pi09='work_df/pi2/common data/09'
pi12='work_df/pi2/common data/12'
pi15='work_df/pi2/common data/15'
pi18='work_df/pi2/common data/18'
#model_path
modelPath='models/XL/r2/weights/best.pt'


value=6
df,t=makeFile(imgPathr2,modelPath,value,1,c_r2)

processing work_df/raw data/round22(renamed)\r2_c1_GOPR0107.JPG
GOPR0107.JPG



0: 640x640 6 Plants, 71.5ms
Speed: 3.0ms preprocess, 71.5ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)



processing work_df/raw data/round22(renamed)\r2_c1_GOPR0108.JPG
GOPR0108.JPG


0: 640x640 6 Plants, 260.0ms
Speed: 2.0ms preprocess, 260.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 Plants, 38.6ms
Speed: 2.0ms preprocess, 38.6ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)


processing work_df/raw data/round22(renamed)\r2_c1_GOPR0109.JPG
GOPR0109.JPG


...


0: 640x640 12 Plants, 41.6ms
Speed: 2.0ms preprocess, 41.6ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)


r2_c3_GOPR0134.JPG
processing work_df/raw data/round22(renamed)\r2_c3_GOPR0135.JPG
GOPR0135.JPG



0: 640x640 12 Plants, 64.8ms
Speed: 2.0ms preprocess, 64.8ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)


r2_c3_GOPR0135.JPG
processing work_df/raw data/round22(renamed)\r2_c3_GOPR0136.JPG
GOPR0136.JPG
r2_c3_GOPR0136.JPG
Total split 141 
 Total single 66


TypeError: cannot unpack non-iterable NoneType object

#### counting correctly predicted images

In [48]:
#making dataframe  
trackingdf=pd.DataFrame(t)
trackingdf.to_csv('S_70.csv')

In [57]:
####################################TO save Images###########################
############################################################################
import shutil

file_image='plant counting/models/s/pictures/pi/18'
file='plant counting/models/s/S_pi18.csv'
img=pi18
# model_path0
df=pd.read_csv(file)
model1 = YOLO('models/s/100/weights/best.pt')
new_df=df[df.T_plant!=df.P_plant]
for i in new_df.image:
    to_path=os.path.join(file_image,i)
    
    model1(source=os.path.join(img,i),save=True,project=file_image)

file=[]
rm=[]
for root,dirs,files in os.walk(file_image):
    for d in dirs:
        for im in os.listdir(os.path.join(root,d)):
            file.append((os.path.join(root,d,im),os.path.join(root,im)))
            rm.append(os.path.join(root,d))
for d,r in zip(file,rm):
    shutil.move(d[0],d[1])
    os.rmdir(r) 


image 1/1 C:\Users\mzr0134\Box\new annotation tool\Test model\work_df\pi2\common data\18\PI_c3_18_03-16-2023 18_04_.jpg: 640x512 11 Plants, 52.3ms
Speed: 3.0ms preprocess, 52.3ms inference, 3.8ms postprocess per image at shape (1, 3, 640, 512)
Results saved to plant counting\models\s\pictures\pi\18\predict

image 1/1 C:\Users\mzr0134\Box\new annotation tool\Test model\work_df\pi2\common data\18\PI_c3_18_03-17-2023 18_05_.jpg: 640x512 12 Plants, 27.1ms
Speed: 2.9ms preprocess, 27.1ms inference, 4.0ms postprocess per image at shape (1, 3, 640, 512)
Results saved to plant counting\models\s\pictures\pi\18\predict2

image 1/1 C:\Users\mzr0134\Box\new annotation tool\Test model\work_df\pi2\common data\18\PI_c3_18_03-23-2023 18_10_.jpg: 640x512 11 Plants, 35.2ms
Speed: 3.0ms preprocess, 35.2ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 512)
Results saved to plant counting\models\s\pictures\pi\18\predict3

image 1/1 C:\Users\mzr0134\Box\new annotation tool\Test model\work_df\

In [51]:
# #making file without function
# path='filetest'
# images=os.listdir(path)
# model_path='runs/segment/train/weights/best.pt'
# model=YOLO(model_path)
# f=[]
# value=6
# for i in images:
    
    
#     if i.split('_')[1]=='c1':
#         r=pred(os.path.join(path,i),model,i.split('_')[-1],value)
        
#         f.append(arrange(r,c1,i.split('_')[-1][:-4],'split'))
#     else:
#         r=pred(os.path.join(path,i),model,i.split('_')[-1],value)
        
#         f.append(arrange(r,c2,i.split('_')[-1][:-4],'split'))
        
# pd.concat(f).to_csv('done.csv')  

Ultralytics YOLOv8.0.28  Python-3.9.13 torch-1.13.1 CUDA:0 (NVIDIA GeForce RTX 3080 Laptop GPU, 8192MiB)


GOPR0047.JPG


YOLOv8x-seg summary (fused): 295 layers, 71721619 parameters, 0 gradients, 343.7 GFLOPs

0: 640x640 6 Plants, 48.5ms
Speed: 0.0ms pre-process, 48.5ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 Plants, 45.1ms
Speed: 0.0ms pre-process, 45.1ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)



GOPR0048 (3).JPG
GOPR0049.JPG


0: 640x640 6 Plants, 38.0ms
Speed: 0.0ms pre-process, 38.0ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 Plants, 38.0ms
Speed: 1.0ms pre-process, 38.0ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 640)


...

GOPR0049.JPG
GOPR0050.JPG



0: 640x640 6 Plants, 39.0ms
Speed: 1.0ms pre-process, 39.0ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 Plants, 40.0ms
Speed: 0.0ms pre-process, 40.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)


GOPR0051.JPG
GOPR0052.JPG



0: 640x640 8 Plants, 39.0ms
Speed: 1.0ms pre-process, 39.0ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 6 Plants, 39.7ms
Speed: 0.0ms pre-process, 39.7ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)


GOPR0053.JPG


### Image information

In [139]:

path=r'work_df/pi2/common data/06'
from PIL import Image
from PIL.ExifTags import TAGS
from datetime import datetime
#####for goPro
def duration(row1,row2):
    print(row1,row2)
    if row1==row2:
        return np.timedelta64(0)
    else: 
        return (row2['Date']-row1['Date'])
    
#for pi
def durationpi(row1,row2):
 
    return (row2['Date']-row1['Date'])

def get_capture_time(image_path):
    try:
        image = Image.open(image_path)
        exif_data = image._getexif()
        
        if exif_data is not None:
            for tag, value in exif_data.items():
                tag_name = TAGS.get(tag, tag)
                if tag_name == 'DateTimeOriginal':
                    return value.split()
                
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None
    
def timedata(path):
    '''
    this function will extract the image information
    path: orginal image path that contain true image details
    output: dataframe for odd and even number images
    
    
    '''
    info={}
    info2={}
    for root,dirs,files in os.walk(path):
        print(root)
        f=0
        while f <len(files)//2:
            print(f' f value is{f}, process {files[f+f]} & {files[f+f+1]}')
            info["r2"+"_c"+root[-1]+"_"+files[f+f]]=get_capture_time(os.path.join(root,files[f+f]))
            info2["r2"+"_c"+root[-1]+"_"+files[f+f+1]]=get_capture_time(os.path.join(root,files[f+f+1]))
        
            f+=1
    d1,d2=pd.DataFrame(info).T,pd.DataFrame(info2).T
    d1.columns,d2.columns=['Date','Time'],['Date','Time']
#     d1['Date'],d2['Date']=d1['Dat'].astype("datetime64[D]"),d2['Data&Time'].astype("datetime64[D]")
#     d1['Date']=d1['Date'].apply(lambda row: duration(d1.iloc[0],row))
#     d2['Date']=d2['Date'].apply(lambda row: duration(d2.iloc[0],row))
    
    return d1,d2
   
    
##############################################################for PI#######################################################


def timedatapi(path_img,path_csv,save=False):
    """
    input: 
        path_img: Images path to collect date information
        path_csv: CSV file that has the image parameter
        
    output: 
        complete dataframe with duration
    """
    info={}
    for files in os.listdir(path_img):
        info[files]=[files[-21:-11]]
        
    date_df=pd.DataFrame(info).T
    
    date_df.reset_index(inplace=True)
    date_df.columns=['Image','Date']
    para_df=pd.read_csv(path_csv)
    
    merged_df=pd.merge(date_df,para_df,on='Image',how='inner')
    
    merged_df["Date"]=pd.to_datetime(merged_df["Date"], format='%m-%d-%Y')
    
    merged_df['Time']=merged_df.apply(lambda row: durationpi(merged_df.iloc[0],row),axis=1)
    
    if save:
        merged_df.to_csv('merged.csv')    
        return merged_df
    else:
        return merged_df
        
    
    
   
   

In [140]:
#########################################For GoPro#########################################

data1,data2=timedata(r'C:\Users\mzr0134\Box\Split Root Experiment\Round_2')
######################################### for PI#########################################
# path_img=r'work_df/pi2/common data/06'
# path_csv='plant counting/pi (common)/parameter/06/06_para.csv'
# data=timedatapi(path_img,path_csv,True)

C:\Users\mzr0134\Box\Split Root Experiment\Round_2
C:\Users\mzr0134\Box\Split Root Experiment\Round_2\Camera 1
 f value is0, process GOPR0107.JPG & GOPR0108.JPG
 f value is1, process GOPR0109.JPG & GOPR0110.JPG
 f value is2, process GOPR0111.JPG & GOPR0112.JPG
 f value is3, process GOPR0113.JPG & GOPR0114.JPG
 f value is4, process GOPR0115.JPG & GOPR0116.JPG
 f value is5, process GOPR0117.JPG & GOPR0118.JPG
 f value is6, process GOPR0119.JPG & GOPR0120.JPG
 f value is7, process GOPR0121.JPG & GOPR0122.JPG
 f value is8, process GOPR0123.JPG & GOPR0124.JPG
 f value is9, process GOPR0125.JPG & GOPR0126.JPG
 f value is10, process GOPR0127.JPG & GOPR0128.JPG
 f value is11, process GOPR0129.JPG & GOPR0130.JPG
 f value is12, process GOPR0131.JPG & GOPR0132.JPG
 f value is13, process GOPR0133.JPG & GOPR0134.JPG
 f value is14, process GOPR0135.JPG & GOPR0136.JPG
 f value is15, process GOPR0137.JPG & GOPR0138.JPG
 f value is16, process GOPR0139.JPG & GOPR0140.JPG
 f value is17, process GOPR0141.

In [144]:

main_df=pd.read_csv('done.csv')

def change_name(df):
    Name=[]
    for i in df['Image']:

        Name.append(i[:-4])
    df['Image']=Name
    return df

main_df=change_name(main_df)
data1.reset_index(inplace=True)
data2.reset_index(inplace=True)

data1=data1.rename(columns={'index':"Image"})
data2=data2.rename(columns={'index':"Image"})

data1=change_name(data1)
data2=change_name(data2)

odd=pd.merge(main_df,data1,on='Image',how='inner')

even=pd.merge(main_df,data2,on='Image',how='inner')
print(f'Shape for ODD {odd.shape}')
print(f'Shape for EVEN {even.shape}')

Shape for ODD (816, 11)
Shape for EVEN (816, 11)


In [162]:
##save dataframe

odd.to_csv('CSVFILE/splited(r2)/odd_full.csv')
even.to_csv('CSVFILE/splited(r2)/even_full.csv')

### Looking into data

In [6]:
def duration(row1,row2):
    if row1.name==row2.name:
        return np.timedelta64(0)
    else: 
        return (row2['Date']-row1['Date'])

dframe1=pd.read_csv('CSVFILE/splited(r2)/odd_full.csv')
dframe1["Date"]=pd.to_datetime(dframe1["Date"], format='%Y:%m:%d')
dframe1.head()


dframe1['Time']=dframe1.apply(lambda row: duration(dframe1.iloc[0],row),axis=1)
dframe1.head()

,Unnamed: 0.1,Unnamed: 0,level_0,axis_major_length,axis_minor_length,area,Plot,Treatment,Image,Condition,Date,Time
0,0,0,1,686.014102,175.026847,43020.0,7,3,r2_c1_GOPR0107,split root,2023-03-10,0 days
1,1,1,4,598.955878,226.100088,42176.0,10,4,r2_c1_GOPR0107,split root,2023-03-10,0 days
2,2,2,3,894.923299,207.106391,48058.0,8,4,r2_c1_GOPR0107,split root,2023-03-10,0 days
3,3,3,5,799.881081,167.016541,38966.0,11,2,r2_c1_GOPR0107,split root,2023-03-10,0 days
4,4,4,2,511.303191,226.166984,43247.0,9,1,r2_c1_GOPR0107,split root,2023-03-10,0 days


In [9]:
dframe1[dframe1['Condition']=='split root'].shape,dframe1[dframe1['Condition']=='single root'].shape

((420, 12), (396, 12))

In [8]:
dframe1.to_csv('CSVFILE/splited(r2)/processedodd1_full.csv')